# 📓 Notebook 3 — Inference, Analysis & Visualization
**Project:** Indonesian Government Law Sentiment Analysis  
**Input:** Fine-tuned XLM-RoBERTa (from Notebook 2) + scraped tweets (from Notebook 0)  
**Output:** Emotion-labeled dataset + full visualization report

---
**Pipeline:**
```
Load model → Clean scraped tweets → Run inference → Map emotion→sentiment
→ Visualize trends → Export final report
```

> ⚠️ Enable GPU: Runtime → Change runtime type → T4 GPU

## ⚙️ Step 1 — Install & Import

In [ ]:
!pip install transformers torch pandas numpy matplotlib seaborn wordcloud emoji scikit-learn -q
print('✅ Packages ready')

In [ ]:
import os
import json
import re
import zipfile
import numpy as np
import pandas as pd
import torch
import emoji
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid', palette='muted')
print(f'✅ Device: {device}')

## 📂 Step 2 — Load Fine-tuned Model

> Upload `xlmroberta_emotion_finetuned.zip` from Notebook 2

In [ ]:
# Unzip model
MODEL_ZIP = 'xlmroberta_emotion_finetuned.zip'
MODEL_DIR = 'xlmroberta_emotion_finetuned'

if not os.path.exists(MODEL_DIR):
    print(f'📦 Unzipping {MODEL_ZIP}...')
    with zipfile.ZipFile(MODEL_ZIP, 'r') as z:
        z.extractall('.')
    print('✅ Unzipped')

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model     = model.to(device)
model.eval()

# Load label mapping
with open(f'{MODEL_DIR}/label_mapping.json', 'r') as f:
    mapping  = json.load(f)
LABEL2ID = mapping['label2id']
ID2LABEL = {int(k): v for k, v in mapping['id2label'].items()}
NUM_LABELS = len(LABEL2ID)

print(f'✅ Model loaded | Labels: {list(LABEL2ID.keys())}')

## 📂 Step 3 — Load Scraped Tweets

> Upload your `tweets_indonesia_law_*.csv` from Notebook 0  
> The cleaner from Notebook 1 is re-applied here automatically

In [ ]:
# ============================================================
# Change filename to match your scraped file
# ============================================================
SCRAPED_CSV = 'tweets_indonesia_law.csv'   # ← update filename

df = pd.read_csv(SCRAPED_CSV)
print(f'✅ Loaded {len(df):,} scraped tweets')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# --- Filter out Grok AI tweets ---
before = len(df)
df = df[df['username'].str.lower() != 'grok'].reset_index(drop=True)
df = df[~df['text'].str.lower().str.startswith('@grok')].reset_index(drop=True)
print(f'🤖 Removed {before - len(df)} Grok AI tweets')
print(f'📊 Remaining: {len(df):,} human tweets')

# --- Parse dates ---
df['created_at'] = pd.to_datetime(df['created_at'])
df['date']       = df['created_at'].dt.date
df['hour']       = df['created_at'].dt.hour

print(f'📅 Date range: {df["created_at"].min().date()} → {df["created_at"].max().date()}')

## 🧹 Step 4 — Clean Scraped Tweets
Re-applying the same cleaning pipeline from Notebook 1

In [ ]:
# Kamus Alay (copy from Notebook 1 — keep in sync if you added entries)
KAMUS_ALAY = {
    'gw': 'saya', 'gue': 'saya', 'w': 'saya',
    'lo': 'kamu', 'lu': 'kamu', 'elo': 'kamu', 'loe': 'kamu',
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
    'enggak': 'tidak', 'kagak': 'tidak', 'ora': 'tidak', 'henteu': 'tidak',
    'tdk': 'tidak', 'blm': 'belum', 'bkn': 'bukan',
    'yg': 'yang', 'dgn': 'dengan', 'dg': 'dengan',
    'krn': 'karena', 'karna': 'karena',
    'utk': 'untuk', 'bgt': 'banget', 'emg': 'memang', 'emang': 'memang',
    'jg': 'juga', 'sdh': 'sudah', 'udah': 'sudah', 'udh': 'sudah',
    'msh': 'masih', 'sm': 'sama', 'ama': 'sama',
    'pd': 'pada', 'dr': 'dari', 'tp': 'tapi', 'tpi': 'tapi',
    'klo': 'kalau', 'kalo': 'kalau', 'klw': 'kalau',
    'knp': 'kenapa', 'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'bs': 'bisa', 'lbh': 'lebih', 'org': 'orang', 'jd': 'jadi',
    'pny': 'punya', 'hrs': 'harus', 'bnyk': 'banyak', 'byk': 'banyak',
    'spt': 'seperti', 'kyk': 'seperti', 'kayak': 'seperti',
    'trs': 'terus', 'trus': 'terus', 'mau': 'mau', 'mo': 'mau',
    'tau': 'tahu', 'tw': 'tahu', 'ntar': 'nanti',
    'sampe': 'sampai', 'ampe': 'sampai', 'dpt': 'dapat',
    'ttg': 'tentang', 'thd': 'terhadap', 'tsb': 'tersebut',
    'iki': 'ini', 'kuwi': 'itu', 'ono': 'ada', 'wis': 'sudah',
    'ane': 'saya', 'ente': 'kamu',
    'wkwk': 'haha', 'wkwkwk': 'haha', 'lol': 'lucu',
    'btw': 'ngomong-ngomong', 'fr': 'sungguh', 'ngl': 'jujur',
    'literally': 'sungguh', 'auto': 'otomatis', 'gaskeun': 'lakukan saja',
    'baper': 'terbawa perasaan', 'kepo': 'ingin tahu',
    # ← add any new entries you found in scraped data here
}

def clean_tweet(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = emoji.demojize(text, language='en', delimiters=(' ', ' '))
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'^rt\s+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    words = text.split()
    text  = ' '.join([KAMUS_ALAY.get(w, w) for w in words])
    text  = re.sub(r'[^\w\s\'-]', ' ', text)
    text  = re.sub(r'\s+', ' ', text).strip()
    return text

print('🔄 Cleaning scraped tweets...')
df['text_clean'] = df['text'].apply(clean_tweet)
df = df[df['text_clean'].str.len() > 5].reset_index(drop=True)
print(f'✅ Done — {len(df):,} tweets after cleaning')

## 🤖 Step 5 — Run Emotion Inference

In [ ]:
def batch_predict(texts, model, tokenizer, device, batch_size=32, max_len=128):
    """
    Run inference on a list of texts.
    Returns: predicted labels, confidence scores, full probability arrays.
    """
    all_preds, all_confs, all_probs = [], [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoding = tokenizer(
            batch,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()

        preds = probs.argmax(axis=1)
        confs = probs.max(axis=1)

        all_preds.extend(preds)
        all_confs.extend(confs)
        all_probs.extend(probs)

        if (i // batch_size + 1) % 10 == 0:
            print(f'  → {i + len(batch)}/{len(texts)} tweets processed...')

    return all_preds, all_confs, all_probs


print('🤖 Running emotion inference...')
texts = df['text_clean'].tolist()
preds, confs, probs = batch_predict(texts, model, tokenizer, device)

# Add results to dataframe
df['emotion']    = [ID2LABEL[p] for p in preds]
df['confidence'] = [round(float(c), 4) for c in confs]

# Add per-emotion probability columns
for i, label in ID2LABEL.items():
    df[f'prob_{label}'] = [round(float(p[i]), 4) for p in probs]

print(f'\n✅ Inference complete!')
print(f'\n=== EMOTION DISTRIBUTION ===')
print(df['emotion'].value_counts())
print(f'\nMean confidence: {df["confidence"].mean():.2%}')

## 🗺️ Step 6 — Map Emotion → Sentiment
Aggregates 6 emotions into 3 sentiment categories for high-level reporting

In [ ]:
# ============================================================
# Emotion → Sentiment mapping
# Adjust this if your research defines it differently
# ============================================================
EMOTION_TO_SENTIMENT = {
    'anger':    'negative',
    'disgust':  'negative',
    'fear':     'negative',
    'sadness':  'negative',
    'joy':      'positive',
    'surprise': 'neutral',   # surprise is ambiguous — could also split to pos/neg
}

SENTIMENT_COLORS = {
    'positive': '#27ae60',
    'negative': '#e74c3c',
    'neutral':  '#95a5a6'
}

EMOTION_COLORS = {
    'anger':    '#e74c3c',
    'joy':      '#f1c40f',
    'fear':     '#9b59b6',
    'disgust':  '#27ae60',
    'sadness':  '#3498db',
    'surprise': '#e67e22'
}

df['sentiment'] = df['emotion'].map(EMOTION_TO_SENTIMENT)

print('=== SENTIMENT DISTRIBUTION ===')
sent_counts = df['sentiment'].value_counts()
sent_pct    = df['sentiment'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'count': sent_counts, 'percent': sent_pct}))

## 📊 Step 7 — Visualizations

In [ ]:
# ── 7A. Overall emotion & sentiment distribution ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Emotion bar
emo_counts = df['emotion'].value_counts()
colors = [EMOTION_COLORS.get(e, '#ccc') for e in emo_counts.index]
bars = axes[0].bar(emo_counts.index, emo_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Emotion Distribution', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Tweet Count')
for bar, val in zip(bars, emo_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontsize=10)

# Sentiment pie
sent_counts = df['sentiment'].value_counts()
s_colors = [SENTIMENT_COLORS[s] for s in sent_counts.index]
axes[1].pie(sent_counts.values, labels=sent_counts.index, colors=s_colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Distribution', fontweight='bold', fontsize=13)

# Confidence distribution
axes[2].hist(df['confidence'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
axes[2].axvline(df['confidence'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["confidence"].mean():.2%}')
axes[2].set_title('Prediction Confidence', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Confidence Score')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('Overall Emotion & Sentiment Analysis\nIndonesian Social Media Law Discourse',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_01_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7B. Emotion by tweet type (original / retweet / reply) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar — emotion by tweet type
pivot = df.groupby(['tweet_type', 'emotion']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(pivot_pct))
emotions_ordered = ['anger', 'disgust', 'fear', 'sadness', 'surprise', 'joy']
for emo in emotions_ordered:
    if emo in pivot_pct.columns:
        axes[0].bar(pivot_pct.index, pivot_pct[emo], bottom=bottom,
                    label=emo, color=EMOTION_COLORS[emo])
        bottom += pivot_pct[emo].values
axes[0].set_title('Emotion by Tweet Type (%)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Percentage')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

# Sentiment by tweet type
sent_pivot = df.groupby(['tweet_type', 'sentiment']).size().unstack(fill_value=0)
sent_pct   = sent_pivot.div(sent_pivot.sum(axis=1), axis=0) * 100
sent_colors = [SENTIMENT_COLORS[s] for s in sent_pct.columns]
sent_pct.plot(kind='bar', ax=axes[1], color=sent_colors, edgecolor='white')
axes[1].set_title('Sentiment by Tweet Type (%)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_02_by_tweet_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7C. Emotion trend over time ────────────────────────────────────────────────
daily = df.groupby(['date', 'emotion']).size().unstack(fill_value=0)
daily.index = pd.to_datetime(daily.index)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Per-emotion line chart
for emo in emotions_ordered:
    if emo in daily.columns:
        axes[0].plot(daily.index, daily[emo], marker='o', markersize=4,
                     label=emo, color=EMOTION_COLORS[emo], linewidth=2)
axes[0].set_title('Daily Emotion Trend', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Tweet Count')
axes[0].legend(ncol=3, fontsize=9)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Stacked area — sentiment over time
daily_sent = df.groupby(['date', 'sentiment']).size().unstack(fill_value=0)
daily_sent.index = pd.to_datetime(daily_sent.index)
for sent in ['negative', 'neutral', 'positive']:
    if sent in daily_sent.columns:
        axes[1].fill_between(daily_sent.index, daily_sent[sent],
                             alpha=0.5, label=sent, color=SENTIMENT_COLORS[sent])
        axes[1].plot(daily_sent.index, daily_sent[sent],
                     color=SENTIMENT_COLORS[sent], linewidth=1.5)
axes[1].set_title('Daily Sentiment Trend', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Tweet Count')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('viz_03_time_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7D. Influencer analysis — emotion by follower tier ────────────────────────
# Segment users by follower count
df['influencer_tier'] = pd.cut(
    df['user_followers'],
    bins=[0, 500, 5000, 50000, float('inf')],
    labels=['Nano\n(<500)', 'Micro\n(500-5K)', 'Mid\n(5K-50K)', 'Macro\n(50K+)']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Emotion by tier
tier_emo = df.groupby(['influencer_tier', 'emotion']).size().unstack(fill_value=0)
tier_pct = tier_emo.div(tier_emo.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(tier_pct))
for emo in emotions_ordered:
    if emo in tier_pct.columns:
        axes[0].bar(tier_pct.index.astype(str), tier_pct[emo], bottom=bottom,
                    label=emo, color=EMOTION_COLORS[emo])
        bottom += tier_pct[emo].values
axes[0].set_title('Emotion by Influencer Tier (%)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Percentage')
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

# Top 10 users by engagement (retweet + like)
df['total_engagement'] = df['retweet_count'] + df['like_count']
top_users = (df.groupby('username')
               .agg(total_engagement=('total_engagement', 'sum'),
                    tweet_count=('tweet_id', 'count'),
                    dominant_emotion=('emotion', lambda x: x.value_counts().index[0]))
               .sort_values('total_engagement', ascending=False)
               .head(10))

bar_colors = [EMOTION_COLORS.get(e, '#ccc') for e in top_users['dominant_emotion']]
axes[1].barh(top_users.index[::-1], top_users['total_engagement'][::-1],
             color=bar_colors[::-1], edgecolor='white')
axes[1].set_title('Top 10 Users by Engagement\n(color = dominant emotion)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Total Likes + Retweets')

plt.tight_layout()
plt.savefig('viz_04_influencers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7E. Word clouds per emotion ────────────────────────────────────────────────
STOPWORDS_ID = {
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk',
    'pada', 'adalah', 'atau', 'juga', 'dalam', 'tidak', 'ada', 'kita',
    'saya', 'kamu', 'dia', 'mereka', 'kami', 'akan', 'sudah', 'belum',
    'bisa', 'harus', 'lebih', 'sangat', 'tapi', 'karena', 'kalau', 'sama',
    'jadi', 'lagi', 'masih', 'seperti', 'ya', 'aja', 'sih', 'nih', 'deh',
    'dong', 'loh', 'saja', 'pun', 'banget', 'mau', 'nanti', 'buat',
    'tahu', 'mungkin', 'memang', 'terus', 'gitu', 'orang',
    'the', 'a', 'an', 'is', 'in', 'on', 'at', 'to', 'of', 'and', 'for',
    'sungguh', 'haha', 'lucu', 'ngomong',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
emo_counts_scraped = df['emotion'].value_counts()

for i, emotion in enumerate(emotions_ordered):
    subset = df[df['emotion'] == emotion]['text_clean']
    if len(subset) == 0:
        axes[i].text(0.5, 0.5, 'No data', ha='center', va='center')
        axes[i].axis('off')
        continue
    text = ' '.join(subset.values)
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        stopwords=STOPWORDS_ID,
        colormap='YlOrRd' if emotion in ['anger', 'disgust'] else
                 'Blues'  if emotion in ['sadness', 'fear'] else
                 'YlGn'   if emotion == 'joy' else 'Oranges',
        max_words=80
    ).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    count = emo_counts_scraped.get(emotion, 0)
    axes[i].set_title(f'{emotion.upper()}  ({count} tweets)',
                       fontsize=13, fontweight='bold',
                       color=EMOTION_COLORS.get(emotion, 'black'))
    axes[i].axis('off')

plt.suptitle('Word Clouds per Emotion — Scraped Law Discourse Tweets',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_05_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7F. Language breakdown ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Language distribution
lang_counts = df['language'].value_counts().head(8)
axes[0].bar(lang_counts.index, lang_counts.values, color='#3498db', edgecolor='white')
axes[0].set_title('Tweet Language Distribution', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')

# Emotion by language (id vs en)
lang_filter = df[df['language'].isin(['id', 'en'])]
lang_emo = lang_filter.groupby(['language', 'emotion']).size().unstack(fill_value=0)
lang_pct = lang_emo.div(lang_emo.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(lang_pct))
for emo in emotions_ordered:
    if emo in lang_pct.columns:
        axes[1].bar(lang_pct.index, lang_pct[emo], bottom=bottom,
                    label=emo, color=EMOTION_COLORS[emo])
        bottom += lang_pct[emo].values
axes[1].set_title('Emotion by Language: id vs en (%)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Percentage')
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

plt.tight_layout()
plt.savefig('viz_06_language.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 Step 8 — Summary Statistics Table

In [ ]:
print('=' * 55)
print('  FINAL ANALYSIS SUMMARY')
print('  Indonesian Social Media Law — Public Reaction')
print('=' * 55)
print(f'  Total tweets analyzed : {len(df):,}')
print(f'  Unique users          : {df["user_id"].nunique():,}')
print(f'  Date range            : {df["created_at"].min().date()} → {df["created_at"].max().date()}')
print(f'  Mean confidence       : {df["confidence"].mean():.2%}')
print()
print('  SENTIMENT BREAKDOWN')
for sent, row in pd.DataFrame({'count': sent_counts, 'pct': sent_pct}).iterrows():
    bar = '█' * int(row['pct'] / 2)
    print(f'  {sent:10s} {bar:25s} {row["count"]:4d} ({row["pct"]:.1f}%)')
print()
print('  EMOTION BREAKDOWN')
for emo, cnt in df['emotion'].value_counts().items():
    pct = cnt / len(df) * 100
    avg_conf = df[df['emotion'] == emo]['confidence'].mean()
    print(f'  {emo:10s}  {cnt:4d} tweets ({pct:5.1f}%)  avg confidence: {avg_conf:.2%}')
print('=' * 55)

## 💾 Step 9 — Save Final Outputs

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- Full labeled dataset (CSV) ---
output_cols = [
    'tweet_id', 'text', 'text_clean', 'created_at', 'date',
    'tweet_type', 'language',
    'emotion', 'sentiment', 'confidence',
    'prob_anger', 'prob_disgust', 'prob_fear', 'prob_joy', 'prob_sadness', 'prob_surprise',
    'like_count', 'retweet_count', 'reply_count',
    'username', 'user_followers', 'user_verified', 'user_location',
    'influencer_tier', 'total_engagement',
    'query_used'
]
# Only include columns that exist
output_cols = [c for c in output_cols if c in df.columns]

final_csv = f'tweets_emotion_labeled_{timestamp}.csv'
df[output_cols].to_csv(final_csv, index=False, encoding='utf-8-sig')
print(f'✅ Saved: {final_csv}')

# --- Excel with multiple sheets ---
final_xlsx = f'tweets_emotion_labeled_{timestamp}.xlsx'
with pd.ExcelWriter(final_xlsx, engine='openpyxl') as writer:
    df[output_cols].to_excel(writer, sheet_name='All Tweets', index=False)
    df[df['sentiment'] == 'negative'][output_cols].to_excel(writer, sheet_name='Negative', index=False)
    df[df['sentiment'] == 'positive'][output_cols].to_excel(writer, sheet_name='Positive', index=False)
    df[df['sentiment'] == 'neutral'][output_cols].to_excel(writer, sheet_name='Neutral', index=False)

    # Summary sheet
    summary = pd.DataFrame({
        'emotion':    df['emotion'].value_counts().index,
        'count':      df['emotion'].value_counts().values,
        'percent':    df['emotion'].value_counts(normalize=True).mul(100).round(1).values,
        'avg_confidence': [df[df['emotion']==e]['confidence'].mean().round(3)
                           for e in df['emotion'].value_counts().index]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)
print(f'✅ Saved: {final_xlsx}')

# --- Download everything ---
from google.colab import files

for fname in [final_csv, final_xlsx,
              'viz_01_overview.png', 'viz_02_by_tweet_type.png',
              'viz_03_time_trend.png', 'viz_04_influencers.png',
              'viz_05_wordclouds.png', 'viz_06_language.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'📥 {fname}')

---
## ✅ Notebook 3 Complete — Full Pipeline Done!

| Visualization | What it shows |
|---|---|
| `viz_01_overview.png` | Overall emotion + sentiment + confidence |
| `viz_02_by_tweet_type.png` | Emotion breakdown by original / retweet / reply |
| `viz_03_time_trend.png` | How emotion & sentiment changed daily |
| `viz_04_influencers.png` | Emotion by follower tier + top engaged users |
| `viz_05_wordclouds.png` | Key words per emotion in law discourse |
| `viz_06_language.png` | Indonesian vs English emotion differences |

| Output file | Contents |
|---|---|
| `tweets_emotion_labeled_*.csv` | Full labeled dataset for further analysis |
| `tweets_emotion_labeled_*.xlsx` | Multi-sheet Excel (by sentiment + summary) |

---
### 🎯 Interpretation Tips
- High **anger + disgust** → strong opposition to the law
- High **fear** → concern about implementation/impact
- High **joy** → support, especially from parents/child advocates
- **Surprise** is ambiguous — read the tweets manually to determine direction
- Compare **influencer tier emotion** — do big accounts frame it differently than regular users?